<a href="https://colab.research.google.com/github/jyotinayak321/IBM-ML-Training/blob/main/MICE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [20]:
#step1 load and pprpare data
df = np.round(pd.read_csv('50_Startups.csv')[
    ['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']
]/10000)

In [21]:
df

,R&D Spend,Administration,Marketing Spend,Profit
0,17.0,14.0,47.0,19.0
1,16.0,15.0,44.0,19.0
2,15.0,10.0,41.0,19.0
3,14.0,12.0,38.0,18.0
4,14.0,9.0,37.0,17.0
5,13.0,10.0,36.0,16.0
6,13.0,15.0,13.0,16.0
7,13.0,15.0,32.0,16.0
8,12.0,15.0,31.0,15.0
9,12.0,11.0,30.0,15.0


In [25]:
#sample 5 rows
np.random.seed(9)
df = df.sample(5)

In [26]:
df


,R&D Spend,Administration,Marketing Spend,Profit
14,12.0,16.0,26.0,13.0
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
44,2.0,15.0,3.0,7.0


In [27]:
#replace target columns (we only impute input feeatures)
df = df.iloc[:,0:-1]


In [28]:
df

,R&D Spend,Administration,Marketing Spend
14,12.0,16.0,26.0
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
44,2.0,15.0,3.0


In [38]:
#step2:Introduce missing values
df.iloc[1, 0] = np.nan   #R&D Speed missing in row 37
df.iloc[3, 1] = np.nan   #Administration missing in row 14
df.iloc[-1, -1] = np.nan #Profit missing in last row


print("original data with missing VALUES")
print(df)

original data with missing VALUES
    R&D Spend  Administration  Marketing Spend
14       12.0            16.0             26.0
21        NaN            15.0             30.0
37        4.0             5.0             20.0
2        15.0             NaN             41.0
44        2.0            15.0              NaN


/tmp/ipykernel_1044/3456592325.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[1, 0] = np.nan   #R&D Speed missing in row 37
/tmp/ipykernel_1044/3456592325.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3, 1] = np.nan   #Administration missing in row 14
/tmp/ipykernel_1044/3456592325.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[-1, -1] = np.nan #Profit missing in last row


# Iteration 0 :mean Imputaion


In [39]:
#interaion with mean imputation
df0 = pd.DataFrame()
df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

print("After Iteration 0 (mean imputation):",df0)


After Iteration 0 (mean imputation):     R&D Spend  Administration  Marketing Spend
14      12.00           16.00            26.00
21       8.25           15.00            30.00
37       4.00            5.00            20.00
2       15.00           12.75            41.00
44       2.00           15.00            29.25


# Iteration 1: First MICE Pass


In [45]:
from sklearn import linear_model
# columns1: Impute R&D Spend
df1 = df0.copy()
df1.iloc[1, 0] = np.nan

# create training data
X_train = df1.iloc[[0, 2, 3, 4], 1:3]
y_train = df1.iloc[[0, 2, 3, 4], 0]

#Train model
lr = LinearRegression()
lr.fit(X_train, y_train)

# predict missing value
X_test = df.iloc[1, 1:].values.reshape(1,2)
prediction = lr.predict(X_test)
df1.iloc[1, 0] = prediction

print(f"prediction R&D Spend for 37: {prediction[0]:.2f}")
#column 2:Impute Administration
df.iloc[3, 1] = np.nan  #restore nan

X_train = df1.iloc[[0, 1, 2, 4], 0:2]   #Cols 1 & 3
y_train = df1.iloc[[0, 1, 2, 4],1]      #Target:administration

prediction R&D Spend for 37: 8.83


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/tmp/ipykernel_1044/1228303297.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.iloc[3, 1] = np.nan  #restore nan
